In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.granularity_utils import (
    measure_3D_granularity,
)

# from granularity import measure_3D_granularity
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    get_mem_and_time_profiling,
)

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-2"
    patient = "NF0014_T1"
    channel = "AGP"
    compartment = "Organoid"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)

In [3]:
channel_mapping = {
    "DNA": "405",
    "AGP": "488",
    "ER": "555",
    "Mito": "640",
    "BF": "TRANS",
    "Nuclei": "nuclei_",
    "Cell": "cell_",
    "Cytoplasm": "cytoplasm_",
    "Organoid": "organoid_",
}

In [4]:
start_time = time.time()
# get starting memory (cpu)
start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_mapping,
    image_set_name=well_fov,
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
if in_notebook:
    verbose = True
else:
    verbose = False
if processor_type == "CPU":
    object_measurements = measure_3D_granularity(
        object_loader=object_loader,
        radius=1,  # radius of the sphere to use for granularity measurement in pixels
        granular_spectrum_length=16,  # usually 16 but 2 is used for testing for now
        subsample_image_value=0.5,  # subsample the image for faster processing. Value should be between 0 and 1. 1 means no subsampling
        z_to_xy_ratio=10.0,  # ratio of z spacing to xy spacing (anisotropy)
        mask_threshold=0.9,  # threshold for determining if an object is too close
        verbose=verbose,
    )
else:
    raise ValueError(
        f"Processor type {processor_type} is not supported. Use 'CPU' only."
    )
final_df = pd.DataFrame(object_measurements)
# get the mean of each value in the array
# melt the dataframe to wide format
final_df = final_df.pivot_table(
    index=["object_id"], columns=["feature"], values=["value"]
)
final_df.columns = final_df.columns.droplevel()
final_df = final_df.reset_index()
# prepend compartment and channel to column names
for col in final_df.columns:
    if col == "object_id":
        continue
    else:
        final_df.rename(
            columns={
                col: format_morphology_feature_name(
                    compartment=compartment,
                    channel=channel,
                    feature_type="Granularity",
                    measurement=f"Granularity_{col}",
                )
            },
            inplace=True,
        )
final_df.insert(0, "image_set", image_set_loader.image_set_name)
output_file = pathlib.Path(
    output_parent_path
    / f"Granularity_{compartment}_{channel}_{processor_type}_features.parquet"
)
output_file.parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(output_file)
final_df.head()

Subsampling image from shape (33, 1537, 1540) with factor 0.5...
  Original shape: (33, 1537, 1540)
  Subsampled shape: (165, 768, 770)
  Unique labels in original: 1
  Unique labels in subsampled: 1
Applying tophat filter...
Start mean: 25.452608281014534
Processing 1 objects


Scale 1 - current_mean: 0.0, prev_mean: 25.452608281014534
  current_mean=0.0, object_mean=25.452608281014534, start_mean=25.452608281014534


Total measurements: 16
Non-zero measurements: 1
Mean granularity: 100.00


feature,image_set,object_id,Organoid_AGP_Granularity_Granularity-1,Organoid_AGP_Granularity_Granularity-2,Organoid_AGP_Granularity_Granularity-3,Organoid_AGP_Granularity_Granularity-4,Organoid_AGP_Granularity_Granularity-5,Organoid_AGP_Granularity_Granularity-6,Organoid_AGP_Granularity_Granularity-7,Organoid_AGP_Granularity_Granularity-8,Organoid_AGP_Granularity_Granularity-9,Organoid_AGP_Granularity_Granularity-10,Organoid_AGP_Granularity_Granularity-11,Organoid_AGP_Granularity_Granularity-12,Organoid_AGP_Granularity_Granularity-13,Organoid_AGP_Granularity_Granularity-14,Organoid_AGP_Granularity_Granularity-15,Organoid_AGP_Granularity_Granularity-16
0,C4-2,1,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
end_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2
end_time = time.time()
get_mem_and_time_profiling(
    start_mem=start_mem,
    end_mem=end_mem,
    start_time=start_time,
    end_time=end_time,
    feature_type="Granularity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Granularity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-2
        Feature type: Granularity
        CPU/GPU: CPU
        Memory usage: 1798.47 MB
        Time elapsed:
        --- 516.25 seconds ---
        --- 8.60 minutes ---
        --- 0.14 hours ---
    


True